# Caviar Strategy Comparison
Compares CSV result files produced by different proving strategies.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from pathlib import Path

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## Configuration
Edit this list to add/remove strategies. Each entry is `(filename, display_name)`.
Paths are relative to `../tmp/`.

In [ ]:
CAVIAR_ROOT = Path(__file__).resolve().parent / "caviar"
TMP = CAVIAR_ROOT / "tmp"

CONFIGS: list[tuple[str, str]] = [
    # ("detour_offset_1.csv", "Detour offset=1"),
    # ("detour_offset_2.csv", "Detour offset=2"),
    ("detour_offset_3.csv", "Detour offset=3"),
    ("detour_offset_9.csv", "Detour offset=9"),
    ("detour_offset_12.csv", "Detour offset=12"),
    # ("detour_offset_6.csv", "Detour offset=6"),
    # ("detour_offset_7.csv", "Detour offset=7"),
    # ("detour_offset_8.csv", "Detour offset=8"),
    # ("detour_offset_9.csv", "Detour offset=9"),
    # ("detour_offset_10.csv", "Detour offset=10"),
    ("results_pulse_0.1.csv", "Pulse threshold=0.1"),
    ("results_prove.csv", "Simple"),
    ("results_npp.csv", "NPP"),
    ("results_pulse_npp_0.25.csv", "PulseNPP threshhold=0.5"),
]

## Load data

In [ ]:
SCHEMA = {
    "index": pl.Int32,
    "start_expression": pl.String,
    "end_expression": pl.String,
    "result": pl.Boolean,
    "best_expr": pl.String,
    "class": pl.Int64,
    "iterations": pl.Int64,
    "egraph_size": pl.Int64,
    "rebuilds": pl.Int64,
    "total_time": pl.Float64,
    "stop_reason": pl.String,
    "condition": pl.String,
    "halide_result": pl.String,
    "halide_time": pl.Float64,
}

frames: dict[str, pl.DataFrame] = {}
for fname, label in CONFIGS:
    path = TMP / fname
    if not path.exists():
        print(f"WARNING: {path} not found, skipping")
        continue
    df = pl.read_csv(path, schema_overrides=SCHEMA, null_values=["", "null", "NULL"])
    frames[label] = df
    print(f"{label:30s}  {len(df):>6} rows  proved={df['result'].sum()}")

assert frames, "No data loaded — check CONFIGS paths"

## 1 Solve rate

In [ ]:
labels = list(frames.keys())
rates = [float(frames[label]["result"].to_numpy().mean()) * 100 for label in labels]
counts = [int(frames[label]["result"].sum()) for label in labels]
totals = [len(frames[label]) for label in labels]

fig, ax = plt.subplots(figsize=(max(5, len(labels) * 1.4), 4))
bars = ax.bar(labels, rates, color="steelblue", width=0.5)
for bar, c, t in zip(bars, counts, totals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{c}/{t}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("Solve rate (%)")
ax.set_ylim(0, 105)
ax.set_title("Solve rate by strategy")
ax.yaxis.set_major_formatter(ticker.PercentFormatter())
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 2 Time to prove (successful only)

In [ ]:
fig, axes = plt.subplots(
    1, len(frames), figsize=(4 * len(frames), 4), sharey=False, squeeze=False
)
axes = axes[0]

for ax, (label, df) in zip(axes, frames.items()):
    proved = df.filter(pl.col("result"))["total_time"]
    if proved.is_empty():
        ax.text(
            0.5,
            0.5,
            "no proved\nexpressions",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
    else:
        times = proved.to_numpy()
        log_times = np.log10(times)
        mean_s = float(times.mean())
        ax.hist(
            log_times,
            bins=30,
            color="seagreen",
            edgecolor="white",
            linewidth=0.4,
        )
        ax.axvline(
            np.log10(mean_s),
            color="red",
            linestyle="--",
            linewidth=1,
            label=f"mean={mean_s:.2f}s",
        )
        ax.legend(fontsize=8)
    ax.set_title(label)
    ax.set_xlabel("log₁₀(Time [s])")
    ax.set_ylabel("Count")

fig.suptitle("Time to prove — successful expressions", fontsize=12)
plt.tight_layout()
plt.show()

## 3 · Total time distribution (all expressions — proved + timeout)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for label, df in frames.items():
    times = df["total_time"].to_numpy()
    ax.ecdf(times, label=label)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF of total_time (all expressions)")
ax.legend()
plt.tight_layout()
plt.show()

## 4 Egraph size distribution

In [ ]:
fig, axes = plt.subplots(
    1, len(frames), figsize=(4 * len(frames), 4), sharey=False, squeeze=False
)
axes = axes[0]

for ax, (label, df) in zip(axes, frames.items()):
    sizes = df["egraph_size"].to_numpy()
    ax.hist(
        np.log10(sizes + 1),
        bins=30,
        color="mediumpurple",
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_title(label)
    ax.set_xlabel("log₁₀(egraph_size + 1)")
    ax.set_ylabel("Count")

fig.suptitle("Egraph size distribution", fontsize=12)
plt.tight_layout()
plt.show()

## 5 Iterations distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for label, df in frames.items():
    iters = df["iterations"].to_numpy()
    ax.ecdf(iters, label=label)

ax.set_xlabel("Iterations")
ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF of iterations")
ax.legend()
plt.tight_layout()
plt.show()

## 6 Summary table

In [ ]:
def _mean(s: pl.Series) -> float:
    a = s.drop_nulls().to_numpy()
    return float(a.mean()) if len(a) else float("nan")


def _median(s: pl.Series) -> float:
    a = s.drop_nulls().to_numpy()
    return float(np.median(a)) if len(a) else float("nan")


rows = []
for label, df in frames.items():
    proved = df.filter(pl.col("result"))
    rows.append(
        {
            "strategy": label,
            "n": len(df),
            "proved": int(df["result"].sum()),
            "prove_rate_%": round(_mean(df["result"]) * 100, 1),
            "time_proved_mean": round(_mean(proved["total_time"]), 3),
            "time_proved_med": round(_median(proved["total_time"]), 3),
            "time_all_mean": round(_mean(df["total_time"]), 3),
            "time_all_med": round(_median(df["total_time"]), 3),
            "egraph_mean": round(_mean(df["egraph_size"]), 1),
            "iter_mean": round(_mean(df["iterations"]), 1),
        }
    )

summary = pl.DataFrame(rows)
display(summary)

## 7 Unique-solve matrix
Cell (A, B) = number of problems solved by A but **not** by B.

In [ ]:
labels = list(frames.keys())
n = len(labels)

# Build a boolean array: solved[i, j] = True if strategy i solved problem j
solved = np.array(
    [frames[label].sort("index")["result"].to_numpy() for label in labels],
    dtype=bool,
)

# matrix[i, j] = # problems solved by i but not j
matrix = np.full((n, n), np.nan)
for i in range(n):
    for j in range(n):
        if i != j:
            matrix[i, j] = int((solved[i] & ~solved[j]).sum())

fig, ax = plt.subplots(figsize=(max(5, n * 1.3), max(4, n * 1.1)))
masked = np.ma.array(matrix, mask=np.isnan(matrix))
im = ax.imshow(masked, cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="problems solved by row, not by column")

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
ax.set_title("Unique-solve matrix\n(row solves, column fails)")

for i in range(n):
    for j in range(n):
        if not np.isnan(matrix[i, j]):
            v = int(matrix[i, j])
            color = (
                "white" if matrix[i, j] > masked.compressed().max() * 0.6 else "black"
            )
            ax.text(j, i, str(v), ha="center", va="center", fontsize=9, color=color)

plt.tight_layout()
plt.show()

## 8 Mean time and node size difference matrices
Cell (A, B) = median(A) − median(B) on problems that **both** A and B solved. Diagonal = 0.

In [ ]:
def _median_diff_matrix(col: str) -> np.ndarray:
    """matrix[i, j] = mean(A[col]) - mean(B[col]) on problems both solved. Diagonal = 0."""
    mat = np.zeros((n, n))
    for i in range(n):
        df_i = frames[labels[i]].sort("index")
        vals_i = df_i[col].to_numpy().astype(float)
        for j in range(n):
            if i == j:
                continue
            df_j = frames[labels[j]].sort("index")
            vals_j = df_j[col].to_numpy().astype(float)
            both = solved[i] & solved[j]
            if both.sum() == 0:
                mat[i, j] = np.nan
                continue
            mat[i, j] = float(np.mean(vals_i[both]) - np.mean(vals_j[both]))
    return mat


def _plot_diff_heatmap(
    ax, mat: np.ndarray, title: str, fmt: str = ".2f", cmap: str = "RdBu_r"
):
    masked = np.ma.array(mat, mask=np.isnan(mat))
    vals = masked.compressed()
    vabs = float(np.abs(vals).max()) if vals.size else 1.0
    im = ax.imshow(masked, cmap=cmap, vmin=-vabs, vmax=vabs)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_title(title)
    for i in range(n):
        for j in range(n):
            if not np.isnan(mat[i, j]):
                color = "white" if abs(mat[i, j]) > vabs * 0.6 else "black"
                ax.text(
                    j,
                    i,
                    format(mat[i, j], fmt),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=color,
                )


time_mat = _median_diff_matrix("total_time")
node_mat = _median_diff_matrix("egraph_size")

fig, axes = plt.subplots(1, 2, figsize=(max(10, n * 2.6), max(4, n * 1.1)))

_plot_diff_heatmap(
    axes[0],
    time_mat,
    "Median total_time (s): row − col\non shared solved problems",
    fmt=".3f",
)
_plot_diff_heatmap(
    axes[1],
    node_mat,
    "Median egraph_size: row − col\non shared solved problems",
    fmt=".0f",
)

plt.tight_layout()
plt.show()